# Parcellation & SLIC Supervoxel Demo

This notebook demonstrates the **anatomically-constrained 3D SLIC** algorithm that
generates supervoxels for graph construction.

## Method Overview

Standard SLIC (Simple Linear Iterative Clustering) is extended with atlas constraints:

- **Eq. 5 (Modified distance):** The distance metric combines spatial distance, intensity
  distance across all modalities, and an atlas penalty term that prevents supervoxels
  from crossing anatomical region boundaries.

- **Eq. 6 (Atlas constraint):** Voxels assigned to different atlas regions receive an
  infinite distance penalty, ensuring each supervoxel is contained within a single
  anatomical region.

This produces supervoxels that are:
- Compact and approximately uniform in size
- Intensity-homogeneous across all MRI modalities
- Anatomically coherent (respect atlas boundaries)

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from stroke_gat.config import Config
from stroke_gat.data.service import DataService
from stroke_gat.slic.atlas_slic import AnatomicalSLIC
from stroke_gat.visualization.parcellation import plot_atlas_slices
from stroke_gat.visualization.supervoxels import (
    plot_supervoxel_slices,
    plot_supervoxel_parcellated_brain,
)

print("Imports successful.")

In [ ]:
# Load configuration and data for one subject
cfg = Config.from_yaml("../configs/default.yaml")
data_service = DataService(cfg)

subjects = data_service.discover_subjects()
subject_id = subjects[0]
print(f"Working with subject: {subject_id}")

# Load multimodal volumes and atlas
volumes = data_service.load_subject(subject_id)
atlas = data_service.load_atlas(subject_id)

print(f"T1 shape: {volumes['T1'].shape}")
print(f"Atlas regions: {len(np.unique(atlas)) - 1}")

In [ ]:
# Visualize the atlas parcellation across axial, coronal, and sagittal views
fig = plot_atlas_slices(
    atlas=atlas,
    background=volumes["T1"],
    title=f"Atlas Parcellation - Subject {subject_id}",
)
plt.show()

In [ ]:
# Run anatomically-constrained SLIC segmentation
slic = AnatomicalSLIC(
    n_supervoxels=cfg.slic.n_supervoxels,
    compactness=cfg.slic.compactness,
    atlas_weight=cfg.slic.atlas_weight,
    max_iterations=cfg.slic.max_iterations,
)

print(f"SLIC parameters:")
print(f"  Target supervoxels: {cfg.slic.n_supervoxels}")
print(f"  Compactness: {cfg.slic.compactness}")
print(f"  Atlas weight: {cfg.slic.atlas_weight}")
print(f"  Max iterations: {cfg.slic.max_iterations}")

# Stack modalities as multi-channel input [H, W, D, C]
multi_modal = np.stack(
    [volumes[m] for m in ["T1", "FLAIR", "ADC", "TRACE"]], axis=-1
)

print(f"\nRunning SLIC on multi-modal volume {multi_modal.shape}...")
labels = slic.fit(multi_modal, atlas=atlas)

n_actual = len(np.unique(labels)) - 1  # exclude background label
print(f"Generated {n_actual} supervoxels (target was {cfg.slic.n_supervoxels})")

In [ ]:
# Plot supervoxel boundaries on axial slices
fig = plot_supervoxel_slices(
    labels=labels,
    background=volumes["T1"],
    n_slices=6,
    title=f"Supervoxel Boundaries - Subject {subject_id}",
)
plt.show()

In [ ]:
# Combined view: atlas regions + supervoxel boundaries
fig = plot_supervoxel_parcellated_brain(
    labels=labels,
    atlas=atlas,
    background=volumes["T1"],
    title=f"Atlas-Constrained Supervoxels - Subject {subject_id}",
)
plt.show()

In [ ]:
# Analyze supervoxel size distribution
unique_labels = np.unique(labels)
unique_labels = unique_labels[unique_labels > 0]  # exclude background

sizes = np.array([np.sum(labels == lbl) for lbl in unique_labels])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Size histogram
axes[0].hist(sizes, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(np.median(sizes), color="red", linestyle="--",
                label=f"Median: {np.median(sizes):.0f} voxels")
axes[0].set_xlabel("Supervoxel Size (voxels)", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)
axes[0].set_title("Supervoxel Size Distribution", fontsize=13)
axes[0].legend(fontsize=11)

# Log-scale for better visibility of tails
axes[1].hist(sizes, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].set_yscale("log")
axes[1].set_xlabel("Supervoxel Size (voxels)", fontsize=12)
axes[1].set_ylabel("Count (log scale)", fontsize=12)
axes[1].set_title("Size Distribution (Log Scale)", fontsize=13)

plt.tight_layout()
plt.show()

print(f"Size statistics:")
print(f"  Min:    {sizes.min():>8,} voxels")
print(f"  Max:    {sizes.max():>8,} voxels")
print(f"  Mean:   {sizes.mean():>8,.1f} voxels")
print(f"  Median: {np.median(sizes):>8,.1f} voxels")
print(f"  Std:    {sizes.std():>8,.1f} voxels")

In [ ]:
# Atlas adherence analysis: what fraction of supervoxels
# are fully contained within a single atlas region?
adherent_count = 0
region_counts = []

for lbl in unique_labels:
    sv_mask = labels == lbl
    atlas_regions_in_sv = np.unique(atlas[sv_mask])
    # Exclude background region (0) from count
    atlas_regions_in_sv = atlas_regions_in_sv[atlas_regions_in_sv > 0]
    n_regions = len(atlas_regions_in_sv)
    region_counts.append(n_regions)
    if n_regions <= 1:
        adherent_count += 1

region_counts = np.array(region_counts)
adherence_rate = 100.0 * adherent_count / len(unique_labels)

fig, ax = plt.subplots(figsize=(8, 5))
bins = np.arange(0.5, region_counts.max() + 1.5, 1)
ax.hist(region_counts, bins=bins, color="darkorange", edgecolor="white", alpha=0.85)
ax.set_xlabel("Atlas Regions per Supervoxel", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(f"Atlas Adherence (single-region rate: {adherence_rate:.1f}%)",
             fontsize=13)
ax.set_xticks(range(1, int(region_counts.max()) + 1))
plt.tight_layout()
plt.show()

print(f"\nAtlas adherence: {adherence_rate:.1f}% of supervoxels span exactly 1 region")

## Discussion

**Supervoxel quality metrics:**

- **Size uniformity:** The histogram shows a relatively tight distribution around the
  median, indicating consistent supervoxel sizes. Some variation is expected due to
  atlas constraints at region boundaries.

- **Atlas adherence:** With the atlas penalty (Eq. 6), nearly all supervoxels are
  contained within a single anatomical region. This ensures that the graph nodes
  have clear anatomical identity.

- **Boundary quality:** The supervoxel boundaries follow both intensity gradients
  (tissue boundaries) and atlas region boundaries, providing a good over-segmentation
  that preserves important structural information.

Next: See `03_graph_construction_demo.ipynb` for building graphs from supervoxels.